# 05 · Evaluación final (prueba única en C5)
Este cuaderno **no reentrena ni vuelve a evaluar**: presenta los resultados de la prueba final que `03_modelado` ejecutó **una sola vez** con el sistema congelado (huella SHA-256 registrada antes de abrir C5).

Contenido: métricas por familia con IC 95 % bootstrap, calibración frente a R/B0/B1, cumplimiento de los criterios del Canvas, explicabilidad, equidad y proyección. Detalle en [`docs/modelado.md`](../docs/modelado.md).

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'yaml': 'PyYAML', 'seaborn': 'seaborn'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title Fuente de resultados
FUENTE = "real"  # @param ["real", "sintetica"]
# "real": agregados de LEMAS ya publicados en results/metrics (autorizados; sin datos individuales).
# "sintetica": resultados de ejecutar 03_modelado con la base sintética.
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from src.utils import cargar_config

config = cargar_config()
MET, FIG = RAIZ / config["rutas"]["metricas"], RAIZ / config["rutas"]["figuras"]
DPI = config["figuras"]["dpi"]
sns.set_theme(style="whitegrid")
COLOR, SUAVE = "#8B1E3F", "#9DB4C9"
print("Fuente:", FUENTE, "| métricas en", MET)

## 1. Resultados en C5

In [ ]:
resultado = json.loads((MET / f"resultados_finales_{FUENTE}.json").read_text(encoding="utf-8"))
print("Sistema congelado (SHA-256):", resultado["sha256_sistema"])
print("Entrenamiento:", resultado["cohortes_entrenamiento"], "| k por sede:", resultado["k_por_sede"])
print("Priorización elegida en C4:", resultado["priorizacion"])
c5 = pd.DataFrame(resultado["C5"])
display(c5)

fig, ax = plt.subplots(figsize=(9, 4))
vista = c5.set_index("sistema")
izq = [v - (vista.loc[s, "ic95_lift_k"] or [v, v])[0] for s, v in vista["lift_k"].items()]
der = [(vista.loc[s, "ic95_lift_k"] or [v, v])[1] - v for s, v in vista["lift_k"].items()]
ax.barh(vista.index, vista["lift_k"], xerr=[izq, der], capsize=4,
        color=[COLOR if s == resultado["priorizacion"] else SUAVE for s in vista.index])
ax.axvline(1, ls="--", c="gray")
ax.set(title="Lift@k en C5 (prueba final) con IC 95 %", xlabel="Lift@k (1 = azar)")
fig.tight_layout(); plt.show()

## 2. Calibración y proyección frente a las líneas base del v5

In [ ]:
display(pd.DataFrame(resultado["brier_C5"]))
print("MAPE por sede y subnivel:", resultado["mape_C5"])
proyeccion = pd.read_csv(MET / f"proyeccion_C5_{FUENTE}.csv")
display(proyeccion)

## 3. Cumplimiento de los criterios del Canvas SMART

In [ ]:
elegido = vista.loc[resultado["priorizacion"]]
reporte = pd.read_csv(MET / f"reporte_6_3_{FUENTE}.csv")
q_familiar = float(reporte.loc[reporte["Cohorte"] == "C5", "q familiar"].iloc[0])
q_estudiante = float(reporte.loc[reporte["Cohorte"] == "C5", "q = U/N"].iloc[0])
brier = {fila["linea_base"]: fila["brier"] for fila in resultado["brier_C5"]}
b1 = next(v for k, v in brier.items() if k.startswith("B1"))
modelo = next(v for k, v in brier.items() if k.startswith("Modelo"))
criterios = pd.DataFrame([
    ["Precision@k familiar > tasa base familiar", f"> {q_familiar:.3f}", elegido["precision_k"],
     elegido["precision_k"] > q_familiar],
    ["Lift@k ≥ 1,20 (límite inferior del IC)", "≥ 1,20", elegido["ic95_lift_k"],
     elegido["lift_k"] >= 1.2],
    ["AP > tasa base estudiantil", f"> {q_estudiante:.3f}", elegido["pr_auc"],
     elegido["pr_auc"] > q_estudiante],
    ["Brier del modelo < B1", f"< {b1}", modelo, modelo < b1],
    ["MAPE ≤ 15 % y menor que B1", "≤ 0,15 y < B1", resultado["mape_C5"],
     resultado["mape_C5"]["modelo"] <= 0.15 and resultado["mape_C5"]["modelo"] < resultado["mape_C5"]["B1"]],
], columns=["criterio", "meta", "resultado", "cumple"])
display(criterios)

## 4. Explicabilidad

In [ ]:
permutacion = pd.read_csv(MET / f"permutacion_{FUENTE}.csv")
shap_imp = pd.read_csv(MET / f"shap_importancia_{FUENTE}.csv")
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
p = permutacion.sort_values("caida_pr_auc")
ejes[0].barh(p["variable"], p["caida_pr_auc"], xerr=p["desv"], color=COLOR)
ejes[0].set(title="Permutación: caída de PR-AUC en C4", xlabel="pérdida al desordenar la variable")
s = shap_imp.head(12).iloc[::-1]
ejes[1].barh(s["variable"], s["shap_medio_abs"], color=SUAVE)
ejes[1].set(title="SHAP medio |valor| (modelo calibrado)", xlabel="efecto medio en el riesgo")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_17_explicabilidad.png", dpi=DPI); plt.show()
print("Lectura: SHAP mide cuánto mueve cada variable la probabilidad; la permutación mide cuánto "
      "ayuda a ORDENAR. Promedio y es_nuevo mueven la probabilidad pero casi no mejoran el orden.")

## 5. Equidad de la priorización (por representante)
Grupos con menos de 5 casos se muestran como `<5`.

In [ ]:
equidad = pd.read_csv(MET / f"equidad_C5_{FUENTE}.csv")
display(equidad)
visibles = equidad.dropna(subset=["recall_k"]).copy()
visibles["etiqueta"] = visibles["variable"] + ": " + visibles["grupo"]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(visibles["etiqueta"], visibles["recall_k"], color=COLOR)
ax.set(title="Recall@k por grupo en C5", xlabel="proporción de familias con no matrícula detectadas")
fig.tight_layout(); fig.savefig(FIG / f"{FUENTE}_18_equidad.png", dpi=DPI); plt.show()

## 6. Conclusiones
1. **La regla D2 se confirma fuera de muestra**: en datos reales, Lift@k = 1,97 [1,24; 2,56]; contactando ~15 % de las familias se llega al ~30 % de las que no se matriculan.
2. **Los modelos de ML no superan a la regla**: la logística híbrida no supera al azar en C5 (0,98 [0,47; 1,52]). La IA sirvió para descubrir, validar y cuantificar la regla.
3. **Calibración y proyección** igualan a la tasa histórica (Brier 0,0648 vs. 0,0647; MAPE 2,8 % en ambos): útiles como referencia, no como mejora.
4. **Equidad**: menor recall en familias becadas (0,18 vs. 0,32; 11 eventos), advertido en la app y en el manual.